# 03. 고성능 모델

- 담당: **③ 안주형 (고성능 모델)** (이 파일은 담당자만 수정)
- 기록 파일: `experiments/log_A_model.csv`

### 할 일
- [ ] XGBoost 개선, LightGBM·CatBoost 비교 (같은 fold, 같은 피처)
- [ ] 클래스 불균형 대응 (`balanced_weights` 등)
- [ ] 튜닝, 앙상블 후보 (cv['oof_proba'] 확률 평균)
- [ ] seed·하이퍼파라미터를 코드에 숫자로 남기기

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [56]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype | DATA_DIR: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\data


In [57]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result, save_predictions, load_predictions
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
from sklearn.metrics import f1_score


OS: Windows-11-10.0.26200-SP0 | python 3.14.7 | pandas 3.1.1 | scikit-learn 1.9.1


## 1. 데이터

In [58]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | classes 26


> **환경 확인:** 위에서 `REPO_DIR`과 `os`를 정의한 후 경로를 출력합니다. 원본에서는 이 셀이 환경 설정보다 앞에 있어 처음부터 실행하면 오류가 발생할 수 있었습니다.

In [59]:
print("현재 작업 경로:", os.getcwd())
print("프로젝트 경로:", REPO_DIR)
print("데이터 폴더:", os.path.join(REPO_DIR, "data"))

현재 작업 경로: c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\notebooks\A_model
프로젝트 경로: c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype
데이터 폴더: c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\data


## 2. 실험 설계 및 데이터 확인

**목적:** 26개 암종 분류를 동일한 Stratified 5-Fold 및 Macro F1로 비교합니다. 모든 모델은 팀 공통 `basic_transformer()`를 Pipeline 내부에서 사용합니다. 변이 문자열은 연속형 유전자 발현량이 아니므로 숫자로 직접 변환하지 않습니다. 테스트 데이터는 최종 추론에서만 사용합니다.

**결과 해석:** Fold 평균과 전체 OOF Macro F1은 서로 다른 집계 방식입니다. 아래 기록은 업로드된 원본 실행 결과이며, 코드 변경 셀은 재실행해야 합니다.

In [60]:
# 변이 데이터 품질 점검: 문자열을 임의의 실수로 변환하지 않음
print("X shape:", X.shape)
print("자료형 분포:")
display(X.dtypes.value_counts())
print("원본 결측값:", int(X.isna().sum().sum()))
print("WT 비율:", round(float(X.eq("WT").to_numpy().mean()), 5))
mutation_binary = binarize_mutations(X)
print("이진 변이 행렬:", mutation_binary.shape)
print("변이 보유 유전자 수(상위 10개):")
display(mutation_binary.sum().sort_values(ascending=False).head(10))
del mutation_binary

X shape: (6201, 4384)
자료형 분포:


str    4384
Name: count, dtype: int64

원본 결측값: 0
WT 비율: 0.99195
이진 변이 행렬: (6201, 4384)
변이 보유 유전자 수(상위 10개):


TP53      1770
PIK3CA     686
RYR2       643
SYNE1      642
PCLO       593
RYR1       473
SPTA1      461
KMT2D      451
IDH1       441
BRAF       438
dtype: int64

## 3. LightGBM 모델 실험

**목적:** 기본 모델(V1), 복잡도 변경(V2), 클래스 불균형 가중치(V3), 가중치 강도(V4)를 동일 Fold로 비교합니다. 소수 암종도 동등하게 반영하기 위해 Macro F1을 사용합니다.

**LightGBM V1 모델 정의:** 팀 공통 이진 변이 전처리와 LightGBM을 Pipeline으로 결합합니다. 모델 설정을 한 곳에서 관리합니다.

In [61]:
from lightgbm import LGBMClassifier

LGBM_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 15,
    "max_depth": 6,
    "random_state": SEED,
    "verbosity": -1,
    "n_jobs": -1
}

lgbm_pipe = Pipeline([
    ("features", basic_transformer()),
    ("model", LGBMClassifier(**LGBM_PARAMS))
])

print("LightGBM 모델 생성 완료")

LightGBM 모델 생성 완료


**V1 검증 결과:** 5-Fold 평균 Macro F1 **0.3030**. 기준 성능으로 보관합니다.

In [62]:
# LightGBM 교차검증
lgbm_cv = run_cv(lgbm_pipe, X, y, folds=folds)

# 결과 확인
display(lgbm_cv)

fold 0: Macro F1 = 0.3114
fold 1: Macro F1 = 0.3068
fold 2: Macro F1 = 0.3115
fold 3: Macro F1 = 0.2901
fold 4: Macro F1 = 0.2954
CV Macro F1 = 0.3030 (std 0.0087)


{'scores': [0.3113808495997006,
  0.30675055269496015,
  0.3114817970343196,
  0.29012893571338166,
  0.29535892444069206],
 'mean': 0.30302021189661077,
 'std': 0.008721174868460777,
 'oof': array([ 2, 19, 20, ...,  4, 18, 20], shape=(6201,)),
 'oof_proba': array([[1.36297420e-04, 2.33843503e-03, 1.22367964e-01, ...,
         4.43644938e-02, 8.27222892e-02, 1.94261857e-02],
        [4.63965341e-05, 2.27709565e-02, 1.08718605e-01, ...,
         2.73507861e-03, 8.54924164e-03, 5.39784339e-03],
        [2.54842200e-04, 2.00567770e-04, 1.43337422e-03, ...,
         2.33614429e-04, 2.43519279e-05, 1.96464058e-04],
        ...,
        [7.91044295e-06, 4.92543521e-04, 8.93177098e-03, ...,
         7.83423460e-03, 1.43953934e-03, 5.46296425e-03],
        [1.42731633e-04, 2.81741421e-03, 1.57850738e-01, ...,
         6.95471099e-02, 4.91538573e-02, 1.64469066e-02],
        [6.17668210e-05, 3.09570350e-04, 9.27040193e-02, ...,
         2.13317802e-02, 1.60839762e-04, 5.65776457e-03]], shape=(6

**V2 실험:** 트리 복잡도와 반복 횟수를 높였습니다. 평균 Macro F1 **0.3004**로 V1보다 낮아 개선안으로 채택하지 않습니다.

In [63]:
from sklearn.base import clone

# 실험 2: LightGBM 하이퍼파라미터 변경
lgbm_v2 = clone(lgbm_pipe)

lgbm_v2.set_params(
    model__n_estimators=400,
    model__learning_rate=0.03,
    model__num_leaves=31,
    model__max_depth=8,
    model__min_child_samples=30,
    model__device_type="cpu"
)

# 기존과 동일한 교차검증
lgbm_cv_v2 = run_cv(lgbm_v2, X, y, folds=folds)

# 성능 비교
print(f"기존 모델 Macro F1: {lgbm_cv['mean']:.4f}")
print(f"개선 실험 Macro F1: {lgbm_cv_v2['mean']:.4f}")
print(f"성능 차이: {lgbm_cv_v2['mean'] - lgbm_cv['mean']:+.4f}")

fold 0: Macro F1 = 0.2901
fold 1: Macro F1 = 0.3166
fold 2: Macro F1 = 0.3030
fold 3: Macro F1 = 0.3059
fold 4: Macro F1 = 0.2867
CV Macro F1 = 0.3004 (std 0.0109)
기존 모델 Macro F1: 0.3030
개선 실험 Macro F1: 0.3004
성능 차이: -0.0026


**V1 클래스별 분석:** 클래스별 precision/recall/F1을 확인해 전체 정확도만으로 놓치기 쉬운 소수 클래스 문제를 살펴봅니다.

In [64]:
from sklearn.metrics import classification_report, confusion_matrix

# 기존 V1 모델의 OOF 예측 결과 분석
print("=== LightGBM V1 클래스별 성능 ===")

print(
    classification_report(
        y,
        lgbm_cv["oof"],
        zero_division=0,
        digits=4
    )
)

print("\n=== 클래스별 실제 데이터 수 ===")
print(pd.Series(y).value_counts().sort_index())

=== LightGBM V1 클래스별 성능 ===
              precision    recall  f1-score   support

           0     0.9000    0.6250    0.7377        72
           1     0.4250    0.1635    0.2361       104
           2     0.3112    0.6489    0.4206       786
           3     0.0909    0.0452    0.0603       155
           4     0.6167    0.6637    0.6393       223
           5     0.6923    0.2368    0.3529        38
           6     0.3457    0.3427    0.3442       461
           7     0.3264    0.2825    0.3029       223
           8     0.1868    0.2311    0.2066       515
           9     0.0837    0.0659    0.0737       334
          10     0.5514    0.3734    0.4453       158
          11     0.2415    0.2489    0.2452       229
          12     0.2759    0.2025    0.2336       158
          13     0.2637    0.1304    0.1745       184
          14     0.3382    0.1292    0.1870       178
          15     0.2601    0.4308    0.3244       253
          16     0.1304    0.0500    0.0723       120

**V3 실험:** LightGBM `class_weight="balanced"`를 사용합니다. 평균 Macro F1 **0.3286**으로 LightGBM 실험 중 가장 높았습니다.

In [65]:
# 실험 3: 클래스 불균형 보정

lgbm_v3 = clone(lgbm_pipe)

lgbm_v3.set_params(
    model__class_weight="balanced",
    model__device_type="cpu"
)

lgbm_cv_v3 = run_cv(
    lgbm_v3,
    X,
    y,
    folds=folds
)

print("\n=== LightGBM 실험 비교 ===")
print(f"V1 기본 모델: {lgbm_cv['mean']:.4f}")
print(f"V2 파라미터 변경: {lgbm_cv_v2['mean']:.4f}")
print(f"V3 클래스 가중치: {lgbm_cv_v3['mean']:.4f}")

fold 0: Macro F1 = 0.3356
fold 1: Macro F1 = 0.3222
fold 2: Macro F1 = 0.3452
fold 3: Macro F1 = 0.3206
fold 4: Macro F1 = 0.3197
CV Macro F1 = 0.3286 (std 0.0101)

=== LightGBM 실험 비교 ===
V1 기본 모델: 0.3030
V2 파라미터 변경: 0.3004
V3 클래스 가중치: 0.3286


**V1·V3 클래스별 비교:** 평균 개선이 모든 암종의 개선을 의미하지는 않으므로 클래스별 F1 차이를 기록합니다.

In [66]:
# V1과 V3의 클래스별 성능 비교
report_v1 = classification_report(
    y, lgbm_cv["oof"],
    output_dict=True,
    zero_division=0
)

report_v3 = classification_report(
    y, lgbm_cv_v3["oof"],
    output_dict=True,
    zero_division=0
)

comparison = pd.DataFrame({
    "클래스": range(26),
    "데이터 수": [
        report_v1[str(i)]["support"] for i in range(26)
    ],
    "V1 F1": [
        report_v1[str(i)]["f1-score"] for i in range(26)
    ],
    "V3 F1": [
        report_v3[str(i)]["f1-score"] for i in range(26)
    ]
})

comparison["F1 변화"] = comparison["V3 F1"] - comparison["V1 F1"]

display(
    comparison.sort_values("F1 변화", ascending=False)
    .round(4)
)

print("V3 전체 OOF Macro F1:",
      round(report_v3["macro avg"]["f1-score"], 4))

,클래스,데이터 수,V1 F1,V3 F1,F1 변화
11,11,229.0,0.2452,0.4399,0.1947
9,9,334.0,0.0737,0.2468,0.1731
24,24,98.0,0.0000,0.1527,0.1527
2,2,786.0,0.4206,0.4945,0.0739
16,16,120.0,0.0723,0.1300,0.0577
17,17,147.0,0.1514,0.2088,0.0574
18,18,266.0,0.1818,0.2299,0.0481
5,5,38.0,0.3529,0.4000,0.0471
1,1,104.0,0.2361,0.2750,0.0389
19,19,198.0,0.1562,0.1932,0.0370


V3 전체 OOF Macro F1: 0.331


**V3 혼동 분석:** 특정 클래스의 과다 예측과 다른 클래스로의 혼동을 확인합니다.

In [67]:
cm_v3 = confusion_matrix(
    y,
    lgbm_cv_v3["oof"],
    labels=np.arange(26)
)

for target_class in [6, 8, 24]:
    row = cm_v3[target_class].copy()
    correct = row[target_class]
    row[target_class] = 0

    top_errors = np.argsort(row)[::-1][:5]

    print(f"\n=== 클래스 {target_class} ===")
    print(f"실제 데이터 수: {cm_v3[target_class].sum()}")
    print(f"정답 예측 수: {correct}")
    print("주요 오분류 대상:")

    for predicted_class in top_errors:
        if row[predicted_class] > 0:
            print(
                f"  클래스 {predicted_class}: "
                f"{row[predicted_class]}건"
            )


=== 클래스 6 ===
실제 데이터 수: 461
정답 예측 수: 101
주요 오분류 대상:
  클래스 11: 200건
  클래스 15: 21건
  클래스 24: 14건
  클래스 2: 14건
  클래스 25: 13건

=== 클래스 8 ===
실제 데이터 수: 515
정답 예측 수: 54
주요 오분류 대상:
  클래스 9: 218건
  클래스 24: 40건
  클래스 17: 36건
  클래스 18: 31건
  클래스 22: 21건

=== 클래스 24 ===
실제 데이터 수: 98
정답 예측 수: 50
주요 오분류 대상:
  클래스 17: 23건
  클래스 8: 3건
  클래스 10: 3건
  클래스 19: 2건
  클래스 18: 2건


**예측 분포 비교:** 클래스별 실제 빈도와 V1·V3 예측 빈도를 비교합니다.

In [68]:
# V1과 V3의 클래스별 예측 분포 비교

prediction_comparison = pd.DataFrame({
    "클래스": np.arange(26),
    "실제 데이터 수": np.bincount(y, minlength=26),
    "V1 예측 수": np.bincount(lgbm_cv["oof"], minlength=26),
    "V3 예측 수": np.bincount(lgbm_cv_v3["oof"], minlength=26),
})

prediction_comparison["V3 예측 변화"] = (
    prediction_comparison["V3 예측 수"]
    - prediction_comparison["V1 예측 수"]
)

prediction_comparison["V3 예측/실제 비율"] = (
    prediction_comparison["V3 예측 수"]
    / prediction_comparison["실제 데이터 수"]
)

display(
    prediction_comparison.sort_values(
        "V3 예측 변화",
        ascending=False
    ).round(3)
)

,클래스,실제 데이터 수,V1 예측 수,V3 예측 수,V3 예측 변화,V3 예측/실제 비율
24,24,98,18,557,539,5.684
17,17,147,104,332,228,2.259
11,11,229,236,353,117,1.541
16,16,120,46,157,111,1.308
22,22,124,104,206,102,1.661
9,9,334,263,363,100,1.087
19,19,198,122,216,94,1.091
18,18,266,196,282,86,1.060
3,3,155,77,159,82,1.026
15,15,253,419,484,65,1.913


**V4 실험:** `0.25 / 0.50 / 0.75`의 부분 가중치를 각 학습 Fold에서만 계산합니다. 결과는 모두 V3의 **0.3286**보다 낮았습니다. 가중치 중복 적용을 방지합니다.

In [69]:
# ============================================================
# V4: 클래스 가중치 강도별 LightGBM 실험
# - 팀 공통 run_cv() 사용
# - 기존 V1 모델의 하이퍼파라미터 유지
# - 각 Fold의 학습 데이터에서만 가중치 계산
# - 평가 지표: 5-Fold CV Macro F1
# ============================================================

from sklearn.utils.class_weight import compute_sample_weight

# 1. 학습 Fold 내부에서 가중치를 계산하는 함수
def make_weight_fn(strength):

    def weight_fn(y_tr):

        balanced = compute_sample_weight(
            class_weight="balanced",
            y=y_tr
        )

        # strength=0: 가중치 없음
        # strength=1: 완전한 balanced
        weights = 1.0 + strength * (balanced - 1.0)

        return {
            "model__sample_weight": weights
        }

    return weight_fn


# 2. 실험할 가중치 강도
weight_strengths = [0.25, 0.50, 0.75]

v4_results = {}
v4_summary = []

# 3. 동일한 모델, 동일한 Fold로 실험
for strength in weight_strengths:

    print("\n" + "=" * 60)
    print(f"V4 실험 | 가중치 강도: {strength}")
    print("=" * 60)

    model = clone(lgbm_pipe)

    # V1의 기본 모델을 사용하며 이중 가중치 적용 방지
    model.set_params(
        model__class_weight=None,
        model__device_type="cpu"
    )

    result = run_cv(
        model,
        X,
        y,
        folds=folds,
        fit_params_fn=make_weight_fn(strength)
    )

    v4_results[strength] = result

    v4_summary.append({
        "모델": f"V4_{strength}",
        "가중치 강도": strength,
        "CV Macro F1": result["mean"],
        "CV Std": result["std"]
    })


# 4. 기존 모델과 비교
comparison_results = pd.DataFrame([
    {
        "모델": "V1",
        "가중치 강도": 0.0,
        "CV Macro F1": lgbm_cv["mean"],
        "CV Std": lgbm_cv["std"]
    },
    {
        "모델": "V3",
        "가중치 강도": 1.0,
        "CV Macro F1": lgbm_cv_v3["mean"],
        "CV Std": lgbm_cv_v3["std"]
    },
    *v4_summary
])

comparison_results = comparison_results.sort_values(
    "CV Macro F1",
    ascending=False
).reset_index(drop=True)

print("\n=== 전체 가중치 실험 비교 ===")
display(comparison_results.round(4))


# 5. 최고 성능 모델 확인
best_row = comparison_results.iloc[0]

print("\n최고 성능 모델:", best_row["모델"])
print("가중치 강도:", best_row["가중치 강도"])
print("CV Macro F1:", round(best_row["CV Macro F1"], 4))


V4 실험 | 가중치 강도: 0.25
fold 0: Macro F1 = 0.3079
fold 1: Macro F1 = 0.3213
fold 2: Macro F1 = 0.3256
fold 3: Macro F1 = 0.3022
fold 4: Macro F1 = 0.3021
CV Macro F1 = 0.3118 (std 0.0098)

V4 실험 | 가중치 강도: 0.5
fold 0: Macro F1 = 0.3149
fold 1: Macro F1 = 0.3170
fold 2: Macro F1 = 0.3459
fold 3: Macro F1 = 0.3119
fold 4: Macro F1 = 0.3222
CV Macro F1 = 0.3224 (std 0.0123)

V4 실험 | 가중치 강도: 0.75
fold 0: Macro F1 = 0.3205
fold 1: Macro F1 = 0.3215
fold 2: Macro F1 = 0.3453
fold 3: Macro F1 = 0.3069
fold 4: Macro F1 = 0.3314
CV Macro F1 = 0.3251 (std 0.0128)

=== 전체 가중치 실험 비교 ===


,모델,가중치 강도,CV Macro F1,CV Std
0,V3,1.00,0.3286,0.0101
1,V4_0.75,0.75,0.3251,0.0128
2,V4_0.5,0.50,0.3224,0.0123
3,V4_0.25,0.25,0.3118,0.0098
4,V1,0.00,0.3030,0.0087



최고 성능 모델: V3
가중치 강도: 1.0
CV Macro F1: 0.3286


**V1·V3·V4 클래스별 비교:** 성능 개선이 일부 클래스에 집중되는지 확인하는 보조 분석입니다.

In [70]:
# V1 / V3 / V4 클래스별 성능 비교
models_to_compare = {
    "V1": lgbm_cv,
    "V4_0.25": v4_results[0.25],
    "V4_0.50": v4_results[0.50],
    "V4_0.75": v4_results[0.75],
    "V3": lgbm_cv_v3
}

rows = []

for model_name, result in models_to_compare.items():
    report = classification_report(
        y, result["oof"],
        output_dict=True,
        zero_division=0
    )

    counts = np.bincount(result["oof"], minlength=26)

    for cls in [6, 8, 9, 11, 17, 24]:
        rows.append({
            "모델": model_name,
            "클래스": cls,
            "실제 수": int(report[str(cls)]["support"]),
            "예측 수": int(counts[cls]),
            "Precision": report[str(cls)]["precision"],
            "Recall": report[str(cls)]["recall"],
            "F1": report[str(cls)]["f1-score"]
        })

detail_comparison = pd.DataFrame(rows)

display(
    detail_comparison.sort_values(
        ["클래스", "모델"]
    ).round(4)
)

,모델,클래스,실제 수,예측 수,Precision,Recall,F1
0,V1,6,461,457,0.3457,0.3427,0.3442
24,V3,6,461,266,0.3797,0.2191,0.2779
6,V4_0.25,6,461,400,0.3400,0.2950,0.3159
12,V4_0.50,6,461,364,0.3489,0.2755,0.3079
18,V4_0.75,6,461,323,0.3375,0.2364,0.2781
1,V1,8,515,637,0.1868,0.2311,0.2066
25,V3,8,515,288,0.1875,0.1049,0.1345
7,V4_0.25,8,515,589,0.1800,0.2058,0.1920
13,V4_0.50,8,515,477,0.1719,0.1592,0.1653
19,V4_0.75,8,515,392,0.1684,0.1282,0.1455


## 4. XGBoost 및 CatBoost 실험

**목적:** 동일한 전처리와 Fold에서 기본 모델과 학습 Fold별 balanced sample weight 모델을 비교합니다. 가중치는 검증 Fold의 정답을 사용하지 않습니다.

**XGBoost V1:** 평균 Macro F1 **0.3201**. LightGBM 기본 모델보다 높으나 V3 balanced보다 낮습니다.

In [71]:
# ============================================================
# XGBoost V1 - 기본 모델
# 기존 LightGBM과 동일한 전처리 / 동일한 5-Fold
# ============================================================

from xgboost import XGBClassifier

# 1. XGBoost 기본 파라미터
XGB_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "multi:softprob",
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "device": "cpu",
    "random_state": SEED,
    "n_jobs": -1
}

# 2. 기존 팀 공통 전처리 사용
xgb_pipe = Pipeline([
    ("features", basic_transformer()),
    ("model", XGBClassifier(**XGB_PARAMS))
])

# 3. 기존과 동일한 5-Fold CV
xgb_cv = run_cv(
    xgb_pipe,
    X,
    y,
    folds=folds
)

# 4. 기존 모델과 성능 비교
print("\n=== 모델 성능 비교 ===")
print(f"LightGBM V1 : {lgbm_cv['mean']:.4f}")
print(f"LightGBM V3 : {lgbm_cv_v3['mean']:.4f}")
print(f"XGBoost V1 : {xgb_cv['mean']:.4f}")

print(
    f"\nXGBoost - LightGBM V3: "
    f"{xgb_cv['mean'] - lgbm_cv_v3['mean']:+.4f}"
)

fold 0: Macro F1 = 0.3137
fold 1: Macro F1 = 0.3305
fold 2: Macro F1 = 0.3367
fold 3: Macro F1 = 0.3193
fold 4: Macro F1 = 0.3005
CV Macro F1 = 0.3201 (std 0.0127)

=== 모델 성능 비교 ===
LightGBM V1 : 0.3030
LightGBM V3 : 0.3286
XGBoost V1 : 0.3201

XGBoost - LightGBM V3: -0.0085


**CatBoost V1:** 평균 Macro F1 **0.2762**. 단독 성능은 낮지만 다른 모델과 오류 양상이 다를 수 있어 앙상블 후보로 보관합니다.

In [72]:
# ============================================================
# CatBoost V1 - 기본 모델
# - 기존 팀 공통 전처리 사용
# - 동일한 5-Fold
# - CPU 학습
# - 기존 코드 수정 없음
# ============================================================

from catboost import CatBoostClassifier

CAT_PARAMS = {
    "iterations": 300,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "MultiClass",
    "l2_leaf_reg": 3,
    "random_seed": SEED,
    "thread_count": -1,
    "task_type": "CPU",
    "verbose": False,
    "allow_writing_files": False
}

cat_pipe = Pipeline([
    ("features", basic_transformer()),
    ("model", CatBoostClassifier(**CAT_PARAMS))
])

# 동일한 5-Fold CV
cat_cv = run_cv(
    cat_pipe,
    X,
    y,
    folds=folds
)

# 결과 비교
comparison = pd.DataFrame([
    {
        "Model": "LightGBM V1",
        "Macro F1": lgbm_cv["mean"],
        "Std": lgbm_cv["std"]
    },
    {
        "Model": "LightGBM V3",
        "Macro F1": lgbm_cv_v3["mean"],
        "Std": lgbm_cv_v3["std"]
    },
    {
        "Model": "XGBoost V1",
        "Macro F1": xgb_cv["mean"],
        "Std": xgb_cv["std"]
    },
    {
        "Model": "CatBoost V1",
        "Macro F1": cat_cv["mean"],
        "Std": cat_cv["std"]
    }
])

display(
    comparison.sort_values(
        "Macro F1",
        ascending=False
    ).round(4)
)

fold 0: Macro F1 = 0.2727
fold 1: Macro F1 = 0.2867
fold 2: Macro F1 = 0.2707
fold 3: Macro F1 = 0.2874
fold 4: Macro F1 = 0.2635
CV Macro F1 = 0.2762 (std 0.0094)


,Model,Macro F1,Std
1,LightGBM V3,0.3286,0.0101
2,XGBoost V1,0.3201,0.0127
0,LightGBM V1,0.3030,0.0087
3,CatBoost V1,0.2762,0.0094


**공통 가중치 함수:** XGBoost와 CatBoost에서 동일한 Fold별 balanced sample weight를 사용합니다. 중복 정의를 하나로 합쳤습니다.

In [73]:
def balanced_fit_params(y_train):
    """훈련 Fold의 라벨만 사용해 가중치 계산."""
    return {"model__sample_weight": compute_sample_weight("balanced", y=y_train)}

**XGBoost V2:** 클래스 가중치 적용 후 5-Fold 평균 Macro F1 **0.3497**로 향상됐습니다. 개별 모델 중 최고 성능입니다.

In [74]:
# ============================================================
# XGBoost V2 - 클래스 불균형 가중치 적용
# 기존 XGBoost V1과 동일한 모델 설정 / Fold
# ============================================================


# 1. 기존 XGBoost 복제
xgb_v2 = clone(xgb_pipe)

# 2. 공통 가중치 함수를 사용합니다.
# 3. 동일한 5-Fold CV
xgb_cv_v2 = run_cv(
    xgb_v2,
    X,
    y,
    folds=folds,
    fit_params_fn=balanced_fit_params
)

# 4. 결과 비교
comparison_v2 = pd.DataFrame([
    {
        "모델": "LightGBM V3",
        "Macro F1": lgbm_cv_v3["mean"],
        "Std": lgbm_cv_v3["std"]
    },
    {
        "모델": "XGBoost V1",
        "Macro F1": xgb_cv["mean"],
        "Std": xgb_cv["std"]
    },
    {
        "모델": "XGBoost V2 Balanced",
        "Macro F1": xgb_cv_v2["mean"],
        "Std": xgb_cv_v2["std"]
    },
    {
        "모델": "CatBoost V1",
        "Macro F1": cat_cv["mean"],
        "Std": cat_cv["std"]
    }
])

display(
    comparison_v2.sort_values(
        "Macro F1",
        ascending=False
    ).round(4)
)

fold 0: Macro F1 = 0.3465
fold 1: Macro F1 = 0.3513
fold 2: Macro F1 = 0.3551
fold 3: Macro F1 = 0.3461
fold 4: Macro F1 = 0.3495
CV Macro F1 = 0.3497 (std 0.0033)


,모델,Macro F1,Std
2,XGBoost V2 Balanced,0.3497,0.0033
0,LightGBM V3,0.3286,0.0101
1,XGBoost V1,0.3201,0.0127
3,CatBoost V1,0.2762,0.0094


**CatBoost V2:** 가중치 적용 후 평균 Macro F1 **0.3189**로 개선됐습니다. 단독 최고 모델은 아니지만 앙상블에 기여하는지 평가합니다.

In [75]:
# ============================================================
# CatBoost V2 - Balanced Sample Weight
# 기존 CatBoost V1과 동일한 설정 및 5-Fold
# ============================================================


# 1. 기존 CatBoost 모델 복제
cat_v2 = clone(cat_pipe)

# 2. 공통 가중치 함수를 사용합니다.
# 3. 동일한 5-Fold CV
cat_cv_v2 = run_cv(
    cat_v2,
    X,
    y,
    folds=folds,
    fit_params_fn=balanced_fit_params
)

# 4. 모든 주요 모델 비교
model_results = {
    "LightGBM V1": lgbm_cv,
    "LightGBM V3 Balanced": lgbm_cv_v3,
    "XGBoost V1": xgb_cv,
    "XGBoost V2 Balanced": xgb_cv_v2,
    "CatBoost V1": cat_cv,
    "CatBoost V2 Balanced": cat_cv_v2
}

comparison_final = pd.DataFrame([
    {
        "모델": name,
        "Macro F1": result["mean"],
        "Std": result["std"]
    }
    for name, result in model_results.items()
])

display(
    comparison_final.sort_values(
        "Macro F1",
        ascending=False
    ).round(4)
)

fold 0: Macro F1 = 0.3055
fold 1: Macro F1 = 0.3157
fold 2: Macro F1 = 0.3219
fold 3: Macro F1 = 0.3284
fold 4: Macro F1 = 0.3230
CV Macro F1 = 0.3189 (std 0.0078)


,모델,Macro F1,Std
3,XGBoost V2 Balanced,0.3497,0.0033
1,LightGBM V3 Balanced,0.3286,0.0101
2,XGBoost V1,0.3201,0.0127
5,CatBoost V2 Balanced,0.3189,0.0078
0,LightGBM V1,0.3030,0.0087
4,CatBoost V1,0.2762,0.0094


## 5. Soft Voting 앙상블 및 오류 분석

**목적:** 서로 다른 모델의 OOF 예측 확률을 평균하여 최종 클래스를 결정합니다. 모든 확률 열은 동일한 클래스 순서여야 합니다. 조합 선택에 같은 OOF를 사용했으므로 최고 점수는 탐색적 결과이며 독립 테스트 검증을 대체하지 않습니다.

**앙상블 조합 비교:** XGB+CAT 전체 OOF Macro F1 **0.3595**, XGB+LGBM+CAT **0.3512**, XGB+LGBM **0.3425**. 동일 가중치 XGB+CAT를 최종 후보로 선택합니다.

In [76]:
# ============================================================
# Ensemble V1: Soft Voting (OOF 기반)
# 기존 학습 결과 재사용, 재학습 없음
# ============================================================

# 1. 각 모델의 OOF 확률
ensemble_models = {
    "XGBoost V2": xgb_cv_v2,
    "LightGBM V3": lgbm_cv_v3,
    "CatBoost V2": cat_cv_v2
}

# 2. OOF 확률 유효성 검사
for name, result in ensemble_models.items():
    proba = result["oof_proba"]

    assert proba is not None, f"{name}: OOF 확률 없음"
    assert proba.shape == (len(y), len(np.unique(y))), (
        f"{name}: OOF 확률 shape 불일치"
    )
    assert np.isfinite(proba).all(), (
        f"{name}: NaN 또는 무한대 존재"
    )

print("모든 모델의 OOF 확률 검증 완료")


# 3. Soft Voting 함수
def soft_voting(results, weights=None):

    probas = [
        result["oof_proba"]
        for result in results
    ]

    avg_proba = np.average(
        np.stack(probas),
        axis=0,
        weights=weights
    )

    predictions = avg_proba.argmax(axis=1)

    return predictions, avg_proba


# 4. 앙상블 조합 비교
ensemble_combinations = {
    "XGB + LGBM": [
        xgb_cv_v2,
        lgbm_cv_v3
    ],
    "XGB + CAT": [
        xgb_cv_v2,
        cat_cv_v2
    ],
    "XGB + LGBM + CAT": [
        xgb_cv_v2,
        lgbm_cv_v3,
        cat_cv_v2
    ]
}

ensemble_results = {}

for name, results in ensemble_combinations.items():

    pred, proba = soft_voting(results)

    score = f1_score(
        y,
        pred,
        average="macro",
        zero_division=0
    )

    ensemble_results[name] = {
        "macro_f1": score,
        "oof": pred,
        "oof_proba": proba
    }


# 5. 단일 모델과 앙상블 비교
comparison_rows = [
    {
        "모델": "XGBoost V2",
        "OOF Macro F1": f1_score(
            y, xgb_cv_v2["oof"],
            average="macro",
            zero_division=0
        )
    },
    {
        "모델": "LightGBM V3",
        "OOF Macro F1": f1_score(
            y, lgbm_cv_v3["oof"],
            average="macro",
            zero_division=0
        )
    },
    {
        "모델": "CatBoost V2",
        "OOF Macro F1": f1_score(
            y, cat_cv_v2["oof"],
            average="macro",
            zero_division=0
        )
    }
]

for name, result in ensemble_results.items():
    comparison_rows.append({
        "모델": name,
        "OOF Macro F1": result["macro_f1"]
    })

ensemble_comparison = pd.DataFrame(
    comparison_rows
).sort_values(
    "OOF Macro F1",
    ascending=False
)

display(ensemble_comparison.round(4))

모든 모델의 OOF 확률 검증 완료


,모델,OOF Macro F1
4,XGB + CAT,0.3595
5,XGB + LGBM + CAT,0.3512
0,XGBoost V2,0.3504
3,XGB + LGBM,0.3425
1,LightGBM V3,0.3310
2,CatBoost V2,0.3191


**앙상블 효과 분석:** XGBoost 단독 대비 정답 전환 **160건**, 오답 전환 **103건**, 순증가 **57건**. 클래스 26개 중 **16개 개선**, **10개 악화**로 개선이 균일하지는 않습니다.

In [77]:
# ============================================================
# 앙상블 분석: XGBoost V2 vs CatBoost V2 vs Soft Voting
# ============================================================

from sklearn.metrics import accuracy_score

y_true = np.asarray(y)

xgb_pred = np.asarray(xgb_cv_v2["oof"])
cat_pred = np.asarray(cat_cv_v2["oof"])
ensemble_pred = np.asarray(ensemble_results["XGB + CAT"]["oof"])

# 1. 각 모델의 정답 여부
xgb_correct = xgb_pred == y_true
cat_correct = cat_pred == y_true
ensemble_correct = ensemble_pred == y_true

print("=" * 60)
print("1. 모델별 정확도 및 Macro F1")
print("=" * 60)

performance = pd.DataFrame([
    {
        "모델": "XGBoost V2",
        "Accuracy": accuracy_score(y_true, xgb_pred),
        "Macro F1": f1_score(
            y_true, xgb_pred,
            average="macro", zero_division=0
        )
    },
    {
        "모델": "CatBoost V2",
        "Accuracy": accuracy_score(y_true, cat_pred),
        "Macro F1": f1_score(
            y_true, cat_pred,
            average="macro", zero_division=0
        )
    },
    {
        "모델": "XGB + CAT",
        "Accuracy": accuracy_score(y_true, ensemble_pred),
        "Macro F1": f1_score(
            y_true, ensemble_pred,
            average="macro", zero_division=0
        )
    }
])

display(performance.round(4))


print("\n" + "=" * 60)
print("2. XGBoost와 CatBoost의 정답 관계")
print("=" * 60)

correctness = pd.DataFrame([
    {
        "구분": "두 모델 모두 정답",
        "샘플 수": int((xgb_correct & cat_correct).sum())
    },
    {
        "구분": "XGBoost만 정답",
        "샘플 수": int((xgb_correct & ~cat_correct).sum())
    },
    {
        "구분": "CatBoost만 정답",
        "샘플 수": int((~xgb_correct & cat_correct).sum())
    },
    {
        "구분": "두 모델 모두 오답",
        "샘플 수": int((~xgb_correct & ~cat_correct).sum())
    }
])

display(correctness)


print("\n" + "=" * 60)
print("3. 앙상블로 개선되거나 악화된 샘플")
print("=" * 60)

improved = (~xgb_correct) & ensemble_correct
worsened = xgb_correct & (~ensemble_correct)

print("XGBoost 오답 → 앙상블 정답:", int(improved.sum()))
print("XGBoost 정답 → 앙상블 오답:", int(worsened.sum()))
print("정답 샘플 순증가:", int(improved.sum() - worsened.sum()))


print("\n" + "=" * 60)
print("4. 클래스별 F1 변화")
print("=" * 60)

xgb_report = classification_report(
    y_true, xgb_pred,
    output_dict=True, zero_division=0
)

ensemble_report = classification_report(
    y_true, ensemble_pred,
    output_dict=True, zero_division=0
)

class_rows = []

for cls in np.unique(y_true):
    key = str(cls)

    xgb_f1 = xgb_report[key]["f1-score"]
    ens_f1 = ensemble_report[key]["f1-score"]

    class_rows.append({
        "클래스": cls,
        "실제 수": int(xgb_report[key]["support"]),
        "XGB F1": xgb_f1,
        "앙상블 F1": ens_f1,
        "F1 변화": ens_f1 - xgb_f1
    })

class_comparison = pd.DataFrame(class_rows)

display(
    class_comparison.sort_values(
        "F1 변화",
        ascending=False
    ).round(4)
)

print(
    "\n개선된 클래스:",
    int((class_comparison["F1 변화"] > 0).sum())
)

print(
    "악화된 클래스:",
    int((class_comparison["F1 변화"] < 0).sum())
)

1. 모델별 정확도 및 Macro F1


,모델,Accuracy,Macro F1
0,XGBoost V2,0.3507,0.3504
1,CatBoost V2,0.3201,0.3191
2,XGB + CAT,0.3599,0.3595



2. XGBoost와 CatBoost의 정답 관계


,구분,샘플 수
0,두 모델 모두 정답,1611
1,XGBoost만 정답,564
2,CatBoost만 정답,374
3,두 모델 모두 오답,3652



3. 앙상블로 개선되거나 악화된 샘플
XGBoost 오답 → 앙상블 정답: 160
XGBoost 정답 → 앙상블 오답: 103
정답 샘플 순증가: 57

4. 클래스별 F1 변화


,클래스,실제 수,XGB F1,앙상블 F1,F1 변화
9,9,334,0.2957,0.3584,0.0627
14,14,178,0.2264,0.2865,0.0601
11,11,229,0.5092,0.5591,0.0500
5,5,38,0.3810,0.4167,0.0357
1,1,104,0.3503,0.3825,0.0322
16,16,120,0.1138,0.1351,0.0213
25,25,198,0.5152,0.5339,0.0187
12,12,158,0.2418,0.2590,0.0172
10,10,158,0.4718,0.4880,0.0162
7,7,223,0.3119,0.3254,0.0134



개선된 클래스: 16
악화된 클래스: 10


**Fold별 안정성:** XGB+CAT 평균 Macro F1 **0.3586**, XGBoost V2 **0.3497**. 5개 Fold 모두 개선됐습니다. pandas의 기본 표본 표준편차(`ddof=1`)와 팀 공통 `run_cv()`의 모집단 표준편차(`ddof=0`)는 다릅니다.

In [78]:
# ============================================================
# Ensemble V2: Fold별 성능 안정성 분석
# - 모델 재학습 없음
# - 기존 OOF 결과 재사용
# - 동일한 5-Fold 검증
# ============================================================


models_for_fold = {
    "XGBoost V2": xgb_cv_v2["oof"],
    "LightGBM V3": lgbm_cv_v3["oof"],
    "CatBoost V2": cat_cv_v2["oof"],
    "XGB + CAT": ensemble_results["XGB + CAT"]["oof"],
    "XGB + LGBM + CAT": ensemble_results[
        "XGB + LGBM + CAT"
    ]["oof"]
}

fold_rows = []

for fold_idx, (train_idx, valid_idx) in enumerate(folds):

    for model_name, predictions in models_for_fold.items():

        score = f1_score(
            np.asarray(y)[valid_idx],
            np.asarray(predictions)[valid_idx],
            average="macro",
            zero_division=0
        )

        fold_rows.append({
            "Fold": fold_idx,
            "Model": model_name,
            "Macro F1": score
        })

fold_results = pd.DataFrame(fold_rows)

print("=== Fold별 성능 ===")

display(
    fold_results.pivot(
        index="Fold",
        columns="Model",
        values="Macro F1"
    ).round(4)
)

print("\n=== 모델별 평균 및 표준편차 ===")

fold_summary = (
    fold_results
    .groupby("Model")["Macro F1"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

display(fold_summary.round(4))

print("\n=== XGBoost 대비 앙상블 Fold별 개선 ===")

fold_pivot = fold_results.pivot(
    index="Fold",
    columns="Model",
    values="Macro F1"
)

fold_pivot["Ensemble Gain"] = (
    fold_pivot["XGB + CAT"]
    - fold_pivot["XGBoost V2"]
)

display(
    fold_pivot[
        ["XGBoost V2", "XGB + CAT", "Ensemble Gain"]
    ].round(4)
)

=== Fold별 성능 ===


Model,CatBoost V2,LightGBM V3,XGB + CAT,XGB + LGBM + CAT,XGBoost V2
Fold,,,,,
0,0.3055,0.3356,0.3578,0.3457,0.3465
1,0.3157,0.3222,0.3535,0.3502,0.3513
2,0.3219,0.3452,0.3634,0.3665,0.3551
3,0.3284,0.3206,0.3613,0.3501,0.3461
4,0.3230,0.3197,0.3569,0.3380,0.3495



=== 모델별 평균 및 표준편차 ===


,mean,std,min,max
Model,,,,
XGB + CAT,0.3586,0.0039,0.3535,0.3634
XGB + LGBM + CAT,0.3501,0.0104,0.3380,0.3665
XGBoost V2,0.3497,0.0037,0.3461,0.3551
LightGBM V3,0.3286,0.0113,0.3197,0.3452
CatBoost V2,0.3189,0.0087,0.3055,0.3284



=== XGBoost 대비 앙상블 Fold별 개선 ===


Model,XGBoost V2,XGB + CAT,Ensemble Gain
Fold,,,
0,0.3465,0.3578,0.0113
1,0.3513,0.3535,0.0022
2,0.3551,0.3634,0.0083
3,0.3461,0.3613,0.0152
4,0.3495,0.3569,0.0073


## 6. 전체 학습 및 테스트 예측

**목적:** 선정된 모델 두 개를 전체 학습 데이터로 다시 학습합니다. 테스트 정답은 사용하지 않습니다. 원본 학습 결과의 테스트 확률 형태는 **(2546, 26)**이며 26개 암종 예측을 생성했습니다. 실제 제출 라벨 변환은 팀 공통 인코더 규칙에 따라 별도 수행해야 합니다.

**최종 추론:** XGBoost V2와 CatBoost V2를 전체 학습 데이터로 학습하고 확률을 1:1 평균합니다. 모델 파라미터와 전처리는 CV 실험과 동일합니다.

In [79]:
# ============================================================
# 최종 후보: XGBoost V2 + CatBoost V2
# 전체 Train 학습 및 Test 예측 확률 생성
# ============================================================


# 1. 최종 모델 복제
final_xgb = clone(xgb_pipe)
final_cat = clone(cat_pipe)

# 2. 전체 학습 데이터 기준 클래스 가중치
full_weights = compute_sample_weight(
    class_weight="balanced",
    y=y
)

# 3. 전체 데이터 학습
print("XGBoost 전체 학습 시작")

final_xgb.fit(
    X,
    y,
    model__sample_weight=full_weights
)

print("XGBoost 전체 학습 완료")

print("\nCatBoost 전체 학습 시작")

final_cat.fit(
    X,
    y,
    model__sample_weight=full_weights
)

print("CatBoost 전체 학습 완료")

# 4. Test 특성 준비
# test는 기존에 로딩한 원본 테스트 DataFrame
X_test = test[X.columns].copy()

assert X_test.columns.equals(X.columns)
assert len(X_test) == len(test)

# 5. 테스트 확률 예측
xgb_test_proba = final_xgb.predict_proba(X_test)
cat_test_proba = final_cat.predict_proba(X_test)

assert xgb_test_proba.shape == cat_test_proba.shape
assert xgb_test_proba.shape[1] == len(np.unique(y))

# 6. 동일 가중치 Soft Voting
final_test_proba = (
    xgb_test_proba + cat_test_proba
) / 2

final_test_pred = final_test_proba.argmax(axis=1)

# 7. 결과 검증
assert np.isfinite(final_test_proba).all()
assert np.allclose(final_test_proba.sum(axis=1), 1, atol=1e-5)

print("\n=== 최종 예측 검증 ===")
print("Test 샘플 수:", len(X_test))
print("확률 행렬:", final_test_proba.shape)
print("최종 예측:", final_test_pred.shape)

print("\n예측 클래스 분포:")
print(
    dict(
        zip(
            *np.unique(
                final_test_pred,
                return_counts=True
            )
        )
    )
)

XGBoost 전체 학습 시작
XGBoost 전체 학습 완료

CatBoost 전체 학습 시작
CatBoost 전체 학습 완료

=== 최종 예측 검증 ===
Test 샘플 수: 2546
확률 행렬: (2546, 26)
최종 예측: (2546,)

예측 클래스 분포:
{np.int64(0): np.int64(24), np.int64(1): np.int64(57), np.int64(2): np.int64(150), np.int64(3): np.int64(58), np.int64(4): np.int64(575), np.int64(5): np.int64(17), np.int64(6): np.int64(64), np.int64(7): np.int64(57), np.int64(8): np.int64(51), np.int64(9): np.int64(73), np.int64(10): np.int64(52), np.int64(11): np.int64(97), np.int64(12): np.int64(45), np.int64(13): np.int64(45), np.int64(14): np.int64(44), np.int64(15): np.int64(169), np.int64(16): np.int64(40), np.int64(17): np.int64(92), np.int64(18): np.int64(43), np.int64(19): np.int64(33), np.int64(20): np.int64(105), np.int64(21): np.int64(115), np.int64(22): np.int64(87), np.int64(23): np.int64(76), np.int64(24): np.int64(250), np.int64(25): np.int64(127)}


**최종 분포 진단 및 오류 수정:** 클래스 4는 학습 비율 **3.60%** 대비 테스트 예측 **22.58%**, 클래스 24는 **1.58%** 대비 **9.82%**로 과다 예측 경향이 있습니다. 단, 테스트 실제 라벨이 없으므로 오류라고 단정할 수 없습니다. **원본 8.09% 모델 간 일치율은 CatBoost 예측의 2차원 형태 때문에 잘못 계산됐을 가능성이 있어, `reshape(-1)`로 수정한 뒤 반드시 재실행해야 합니다.**

In [80]:
# ============================================================
# 최종 예측 진단
# ============================================================

train_dist = (
    pd.Series(np.asarray(y))
    .value_counts()
    .reindex(range(26), fill_value=0)
)

test_dist = (
    pd.Series(final_test_pred)
    .value_counts()
    .reindex(range(26), fill_value=0)
)

distribution_check = pd.DataFrame({
    "클래스": range(26),
    "Train 실제 수": train_dist.values,
    "Train 비율(%)": train_dist.values / len(y) * 100,
    "Test 예측 수": test_dist.values,
    "Test 예측 비율(%)": test_dist.values / len(final_test_pred) * 100
})

distribution_check["비율 차이(%p)"] = (
    distribution_check["Test 예측 비율(%)"]
    - distribution_check["Train 비율(%)"]
)

display(distribution_check.round(2))

# 개별 모델과 앙상블의 예측 비교
xgb_pred_test = np.asarray(final_xgb.predict(X_test)).reshape(-1)
cat_pred_test = np.asarray(final_cat.predict(X_test)).reshape(-1)

assert xgb_pred_test.shape == cat_pred_test.shape == final_test_pred.shape

print("XGBoost / CatBoost 예측 일치율:",
      round(np.mean(xgb_pred_test == cat_pred_test), 4))

print("XGBoost / 앙상블 예측 일치율:",
      round(np.mean(xgb_pred_test == final_test_pred), 4))

# 클래스별 확률 열 순서 검증
expected_classes = np.arange(26)

assert np.array_equal(
    final_xgb.named_steps["model"].classes_,
    expected_classes
)

assert np.array_equal(
    final_cat.named_steps["model"].classes_,
    expected_classes
)

print("클래스 확률 열 순서 검증 완료")

# 예측 확률과 직접 계산한 argmax 결과 일치 여부 확인
assert np.array_equal(xgb_pred_test, xgb_test_proba.argmax(axis=1))
assert np.array_equal(cat_pred_test, cat_test_proba.argmax(axis=1))
print("모델별 predict / predict_proba 정렬 검증 완료")


,클래스,Train 실제 수,Train 비율(%),Test 예측 수,Test 예측 비율(%),비율 차이(%p)
0,0,72,1.16,24,0.94,-0.22
1,1,104,1.68,57,2.24,0.56
2,2,786,12.68,150,5.89,-6.78
3,3,155,2.50,58,2.28,-0.22
4,4,223,3.60,575,22.58,18.99
5,5,38,0.61,17,0.67,0.05
6,6,461,7.43,64,2.51,-4.92
7,7,223,3.60,57,2.24,-1.36
8,8,515,8.31,51,2.00,-6.30
9,9,334,5.39,73,2.87,-2.52


XGBoost / CatBoost 예측 일치율: 0.6371
XGBoost / 앙상블 예측 일치율: 0.8778
클래스 확률 열 순서 검증 완료
모델별 predict / predict_proba 정렬 검증 완료


## 7. 결론 및 재현성 점검

- **최종 후보:** XGBoost V2 + CatBoost V2 (동일 가중치 Soft Voting)
- **검증:** 5-Fold 평균 Macro F1 **0.3586**, 전체 OOF Macro F1 **0.3595** (업로드 원본 실행 기록)
- **기준 모델:** XGBoost V2 평균 **0.3497**, 전체 OOF **0.3504**
- **한계:** 동일 OOF를 여러 실험과 모델 선택에 반복 활용했으므로 독립 성능 추정치가 아님. 테스트의 클래스 분포도 정답 분포를 뜻하지 않음
- **재현:** 팀 공통 `src` 모듈, 원본 `train.csv`/`test.csv`, 동일 패키지 버전 및 `SEED`가 필요합니다. 본 파일은 업로드된 실행 기록을 보존했으나 수정 셀과 전체 노트북을 실제 환경에서 처음부터 재실행해 검증해야 합니다.
- **팀 공유:** 공통 코드의 변경 없이 담당 모델링 셀을 정리했습니다. 최종 제출 전 클래스 라벨 복원 및 제출 스키마를 팀 담당자와 확인하세요.

## 8. 11월 7일 실험 기록 — 팀 공통 함수 사용

**목적:** `run_cv()`로 이미 얻은 결과를 `src.evaluation.log_result()`로 `experiments/log_A_model.csv`에 저장합니다. 원본 실험·모델 파라미터·CV 결과는 변경하지 않습니다.

- 기록 날짜는 **코드를 실행한 실제 시각**입니다. 실험 식별자에는 `1107`을 포함합니다.
- `log_result()`는 CSV에 **추가(append)** 하므로 같은 실험 ID가 있으면 재기록하지 않습니다.
- XGBoost/CatBoost V2는 **기본 모델 파라미터 + Fold별 `balanced_fit_params()`**를 사용했습니다. 모델 객체의 파라미터만 기록하면 가중치 설정이 누락되므로 `notes`에도 명시합니다.
- LightGBM V4는 각 가중치 강도마다 별도 실험으로 기록합니다.
- **기록은 5-Fold 전체 실행 결과만** 허용합니다. 이 셀은 재학습하지 않습니다.
- 기존 LightGBM CV는 `deterministic=True, force_row_wise=True`가 없는 설정으로 실행되었습니다. 재현성 옵션을 적용한 새 실험은 별도 ID로 수행해야 하며, 기존 점수를 새 설정의 점수로 기록하면 안 됩니다.


In [81]:
# 11/07 실험 결과를 팀 공통 CSV에 기록 (재실행 시 중복 방지)
# 사전 조건: 위의 5-Fold CV 셀들을 정상 실행했을 것
from src.config import EXPERIMENT_DIR

OWNER = "A_model"
LOG_PATH = EXPERIMENT_DIR / f"log_{OWNER}.csv"

# 원본 CV 결과의 실제 모델 설정을 그대로 기록합니다.
experiment_specs = [
    ("1107_lgbm_v1", "LightGBM", lgbm_pipe, lgbm_cv, "기본 모델"),
    ("1107_lgbm_v2", "LightGBM", lgbm_v2, lgbm_cv_v2, "n_estimators=400, learning_rate=0.03, num_leaves=31, max_depth=8, min_child_samples=30"),
    ("1107_lgbm_v3", "LightGBM", lgbm_v3, lgbm_cv_v3, "class_weight=balanced"),
    ("1107_xgb_v1", "XGBoost", xgb_pipe, xgb_cv, "기본 모델"),
    ("1107_xgb_v2", "XGBoost", xgb_v2, xgb_cv_v2, "Fold 학습 라벨 기준 balanced sample_weight; fit_params_fn=balanced_fit_params"),
    ("1107_cat_v1", "CatBoost", cat_pipe, cat_cv, "기본 모델"),
    ("1107_cat_v2", "CatBoost", cat_v2, cat_cv_v2, "Fold 학습 라벨 기준 balanced sample_weight; fit_params_fn=balanced_fit_params"),
]

for strength in (0.25, 0.50, 0.75):
    experiment_specs.append((
        f"1107_lgbm_v4_{int(strength * 100):03d}",
        "LightGBM", lgbm_pipe, v4_results[strength],
        f"Fold 학습 라벨 기준 부분 가중치 strength={strength}; class_weight=None; device_type=cpu"
    ))

existing_ids = set()
if LOG_PATH.exists() and LOG_PATH.stat().st_size > 0:
    existing_df = pd.read_csv(LOG_PATH, encoding="utf-8-sig")
    if "experiment" not in existing_df.columns:
        raise ValueError(f"기존 로그에 experiment 열이 없습니다: {LOG_PATH}")
    existing_ids = set(existing_df["experiment"].dropna().astype(str))

for exp_id, model_name, pipe, cv_result, note in experiment_specs:
    if exp_id in existing_ids:
        print(f"기록 유지(중복 생략): {exp_id}")
        continue
    if len(cv_result["scores"]) != len(folds) or len(folds) != 5:
        raise ValueError(f"{exp_id}: 5-Fold 전체 결과만 기록할 수 있습니다.")
    params = pipe.named_steps["model"].get_params().copy()
    if exp_id.startswith("1107_lgbm_v4"):
        params.update(class_weight=None, device_type="cpu", weight_strength=float(note.split("strength=")[1].split(";")[0]))
    if exp_id.endswith("xgb_v2") or exp_id.endswith("cat_v2"):
        params["fit_sample_weight"] = "balanced_per_training_fold"
    log_result(
        owner=OWNER,
        notebook="notebooks/A_model/03_models.ipynb",
        experiment=exp_id,
        features="WT_binary (basic_transformer)",
        model=model_name,
        params=params,
        cv=cv_result,
        notes=f"11/07 실험 | seed={SEED} | {note}"
    )
    existing_ids.add(exp_id)

print("\n현재 기록된 실험 결과:")
display(pd.read_csv(LOG_PATH, encoding="utf-8-sig")[["experiment", "model", "cv_mean", "cv_std", "notes"]])

기록: log_A_model.csv ← 1107_lgbm_v1 (CV 0.3030)
기록: log_A_model.csv ← 1107_lgbm_v2 (CV 0.3004)
기록: log_A_model.csv ← 1107_lgbm_v3 (CV 0.3286)
기록: log_A_model.csv ← 1107_xgb_v1 (CV 0.3201)
기록: log_A_model.csv ← 1107_xgb_v2 (CV 0.3497)
기록: log_A_model.csv ← 1107_cat_v1 (CV 0.2762)
기록: log_A_model.csv ← 1107_cat_v2 (CV 0.3189)
기록: log_A_model.csv ← 1107_lgbm_v4_025 (CV 0.3118)
기록: log_A_model.csv ← 1107_lgbm_v4_050 (CV 0.3224)
기록: log_A_model.csv ← 1107_lgbm_v4_075 (CV 0.3251)

현재 기록된 실험 결과:


,experiment,model,cv_mean,cv_std,notes
0,1107_lgbm_v1,LightGBM,0.3030,0.0087,11/07 실험 | seed=42 | 기본 모델
1,1107_lgbm_v2,LightGBM,0.3004,0.0109,"11/07 실험 | seed=42 | n_estimators=400, learnin..."
2,1107_lgbm_v3,LightGBM,0.3286,0.0101,11/07 실험 | seed=42 | class_weight=balanced
3,1107_xgb_v1,XGBoost,0.3201,0.0127,11/07 실험 | seed=42 | 기본 모델
4,1107_xgb_v2,XGBoost,0.3497,0.0033,11/07 실험 | seed=42 | Fold 학습 라벨 기준 balanced sa...
5,1107_cat_v1,CatBoost,0.2762,0.0094,11/07 실험 | seed=42 | 기본 모델
6,1107_cat_v2,CatBoost,0.3189,0.0078,11/07 실험 | seed=42 | Fold 학습 라벨 기준 balanced sa...
7,1107_lgbm_v4_025,LightGBM,0.3118,0.0098,11/07 실험 | seed=42 | Fold 학습 라벨 기준 부분 가중치 stre...
8,1107_lgbm_v4_050,LightGBM,0.3224,0.0123,11/07 실험 | seed=42 | Fold 학습 라벨 기준 부분 가중치 stre...
9,1107_lgbm_v4_075,LightGBM,0.3251,0.0128,11/07 실험 | seed=42 | Fold 학습 라벨 기준 부분 가중치 stre...


## 9. 앙상블 후보 확률 저장

**목적:** `05_ensemble.ipynb`에서 같은 행·같은 클래스 순서의 OOF 확률을 사용할 수 있도록 **XGBoost V2와 CatBoost V2**의 OOF/Test 확률을 팀 표준 형식으로 저장합니다.

- OOF 확률은 각 검증 Fold에서만 예측한 값이며, 테스트 확률은 **전체 Train으로 학습한 동일 설정의 모델**에서 얻은 값입니다.
- `le.classes_`는 원래 SUBCLASS 문자열의 순서이며, 모델 확률의 열 순서(인코딩된 0~25)와 대응합니다.
- 기존 파일명이 같으면 덮어쓰므로 **최종 후보의 확률을 저장할 때만** 실행하세요.
- `experiments/preds/`는 팀 README상 공유 대상입니다. 단, 파일 크기·대회 규정·`.gitignore`를 커밋 전에 확인하세요.
- 테스트 예측 분포를 보고 모델 가중치나 파라미터를 변경하지 않습니다.


In [82]:
# 앙상블 후보 모델의 OOF/Test 확률 저장 (재학습 없음)
class_ids = np.arange(len(le.classes_))
assert np.array_equal(np.unique(y), class_ids), "타깃 인코딩이 0..K-1인지 확인하세요"
assert np.array_equal(final_xgb.named_steps["model"].classes_, class_ids)
assert np.array_equal(final_cat.named_steps["model"].classes_, class_ids)

for name, cv_result, test_proba in [
    ("xgb_v2_balanced_1107", xgb_cv_v2, xgb_test_proba),
    ("cat_v2_balanced_1107", cat_cv_v2, cat_test_proba),
]:
    oof = np.asarray(cv_result["oof_proba"])
    test_probs = np.asarray(test_proba)
    assert oof.shape == (len(y), len(class_ids)), f"{name}: OOF shape 오류"
    assert test_probs.shape == (len(test), len(class_ids)), f"{name}: Test shape 오류"
    assert np.isfinite(oof).all() and np.isfinite(test_probs).all()
    assert np.allclose(oof.sum(axis=1), 1, atol=1e-4)
    assert np.allclose(test_probs.sum(axis=1), 1, atol=1e-4)
    save_predictions(
        owner="A_model",
        name=name,
        oof_proba=oof,
        test_proba=test_probs,
        classes=le.classes_,
        cv=cv_result,
        notes="2026-11-07 | WT_binary | balanced sample_weight | 5-Fold | CPU"
    )

# 저장된 두 후보를 읽어 파일/클래스/행 수가 올바른지 확인
saved = load_predictions(classes=le.classes_)
for name in ("xgb_v2_balanced_1107", "cat_v2_balanced_1107"):
    key = f"A_model__{name}"
    assert key in saved
    assert saved[key]["oof"].shape == (len(y), len(class_ids))
    assert saved[key]["test"].shape == (len(test), len(class_ids))
print("앙상블 후보 2개 저장 및 재로딩 확인 완료")

예측 저장: A_model__xgb_v2_balanced_1107 (CV 0.3497) → experiments/preds/
예측 저장: A_model__cat_v2_balanced_1107 (CV 0.3189) → experiments/preds/
불러온 예측 3개: ['A_model__cat_v2_balanced_1107', 'A_model__xgb_v2_balanced_1107', 'J_leader__baseline_xgb']
앙상블 후보 2개 저장 및 재로딩 확인 완료


## 10. 제출 전 체크리스트

1. **Kernel 재시작 → Run All**로 5-Fold 결과와 로그 저장까지 오류 없이 재현되는지 확인합니다.
2. `experiments/log_A_model.csv`의 모델별 CV 값이 실제 실행 출력과 일치하는지 확인합니다. `log_result()`의 `date`는 기록 실행 시각입니다.
3. `experiments/preds/`의 OOF/Test 확률 및 메타데이터 파일이 정상 생성됐는지 확인합니다.
4. 이 노트북에서 `src/` 공통 코드를 수정하지 않았습니다. `src` 변경이 필요하면 담당자와 공유합니다.
5. **원본 train/test CSV, 모델 파일, 제출 CSV, 가상환경은 커밋하지 않습니다.**
6. `feat/model_A_1107`에서 본인 노트북과 `experiments/log_A_model.csv`, 담당자 예측 파일만 스테이징한 뒤 `dev` 대상으로 PR을 올립니다. 팀원 1명 승인 후 Merge합니다.

> 기존 결과 수치는 과거 실행 기록이며, 이 파일을 제작하면서 실제 데이터로 전체 학습을 다시 수행한 것은 아닙니다. 재실행 후 결과를 확인하세요.
